# 008 CIVD Within-Cluster Equal-Split Problem Analysis

**Goal**: compare the allocation results of Voronoi and CIVD under the best static weighting `prox2_ntl_landuse_demand`,
exposing CIVD's core weakness — equal-split demand within a cluster.

**Method**:
- `voronoi_prox2_ntl_gpm`: Voronoi allocation + GPM + NTL + Proximity(γ=2) correction
- `civd_prox2_ntl_gpm`: CIVD allocation (equal split within cluster) + the same correction

**Data source**: uses the CIVD assignment already cached by the 003 notebook (`results/static_allocation/`)

In [ ]:
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROJECT_ROOT = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from SpatialAllocation.Allocator import allocator_registry
from SpatialAllocation.Allocator.clustering.do_clustering import do_clustering
from SpatialAllocation.Weighter import weighter_registry
from SpatialAllocation.FeatureExtractor.correctors import corrector_registry
from SpatialAllocation.FeatureExtractor.correctors.proximity_corrector import ProximityCorrector

warnings.filterwarnings('ignore', category=FutureWarning)

# ─── Path constants ───
DATA_DIR = Path('./results/intermediate')
ASSEMBLED_DIR = DATA_DIR / 'features' / 'assembled'
EXTRACTED_DIR = DATA_DIR / 'features' / 'extracted'
CACHE_DIR = Path('./results/static_allocation')

GAMMA = 2.0
TARGET_CRS = 'EPSG:27700'
DIST_CLAMP_KM = 0.01

STUDY_REGIONS = [
    'London',
    'TLH2', 'TLH3', 'TLJ1', 'TLF1', 'TLF2',
    'TLC1', 'TLC2', 'TLD6', 'TLG1', 'TLG2', 'TLE4',
    'TLH1', 'TLE3', 'TLD3', 'TLD4',
]

LANDUSE_PERCENT_MAP = {
    'lu_residential_prop': 'residential_percent',
    'lu_commercial_prop': 'commercial_percent',
    'lu_industrial_prop': 'industrial_percent',
    'lu_agricultural_prop': 'agricultural_percent',
    'lu_others_prop': 'others_percent',
}
LU_COLS = list(LANDUSE_PERCENT_MAP.keys())
PCT_COLS = list(LANDUSE_PERCENT_MAP.values())

In [ ]:
# ─── Load data ───
region_gdf = gpd.read_file(str(DATA_DIR / 'ITL3_region.gpkg'))
substations_gdf = gpd.read_file(str(DATA_DIR / 'substations.gpkg'))

grids = {}
ntl_data = {}

for loc in STUDY_REGIONS:
    path = ASSEMBLED_DIR / f'{loc}_grid_points.pickle'
    with open(path, 'rb') as f:
        grid_gdf, step_size_m = pickle.load(f)
    grids[loc] = (grid_gdf, step_size_m)

    ntl_path = EXTRACTED_DIR / f'{loc}_ntl.npz'
    ntl_npz = np.load(ntl_path, allow_pickle=True)
    ntl_data[loc] = ntl_npz['data'][:, 0]

print(f'Loaded {len(grids)} regions')

In [ ]:
# ─── Helper functions ───

def compute_demand(grid_gdf, region_sub, weighter_result, demand_col='demand'):
    W = weighter_result.weights
    gdf = grid_gdf.copy()
    gdf[demand_col] = 0.0
    region_info = region_sub.set_index('ITL3')
    for itl3, group in gdf.groupby('ITL3'):
        if itl3 not in region_info.index:
            continue
        total_demand = region_info.loc[itl3, 'Demand (MVA)']
        idx = group.index
        if W.ndim == 2:
            pcts = np.array([region_info.loc[itl3, c] for c in PCT_COLS])
            score = W[idx] @ pcts
        else:
            score = W[idx]
        score_sum = score.sum()
        if score_sum > 0:
            gdf.loc[idx, demand_col] = total_demand * score / score_sum
        else:
            gdf.loc[idx, demand_col] = total_demand / len(group)
    return gdf


def aggregate_to_substations(grid_gdf, subs_gdf, assignment, demand_col):
    result = subs_gdf.copy()
    result['allocated_demand'] = 0.0
    demands = grid_gdf[demand_col].values
    for target_idx in range(len(subs_gdf)):
        mask = assignment == target_idx
        result.loc[target_idx, 'allocated_demand'] = demands[mask].sum()
    return result


def aggregate_clustered_to_substations(grid_gdf, subs_gdf, cluster_gdf,
                                        assignment, demand_col):
    result = subs_gdf.copy()
    result['allocated_demand'] = 0.0
    demands = grid_gdf[demand_col].values
    cluster_demands = {}
    for label in np.unique(assignment):
        mask = assignment == label
        cluster_demands[label] = demands[mask].sum()
    for label, total_d in cluster_demands.items():
        members = cluster_gdf[cluster_gdf['cluster_label'] == label].index
        n_members = len(members)
        if n_members > 0:
            for idx in members:
                if idx < len(result):
                    result.loc[idx, 'allocated_demand'] += total_d / n_members
    return result


def evaluate_allocation(subs_result, actual_col='Demand (MVA)',
                        alloc_col='allocated_demand'):
    actual = subs_result[actual_col].values
    allocated = subs_result[alloc_col].values
    corr, _ = pearsonr(actual, allocated)
    rmse = np.sqrt(mean_squared_error(actual, allocated))
    mae = mean_absolute_error(actual, allocated)
    return {'corr': corr, 'rmse': rmse, 'mae': mae}

In [ ]:
# ─── Main loop: compute the best demand + load cached assignment + evaluate ───

ntl_corrector = corrector_registry.create('ntl')
prox_corrector = corrector_registry.create('proximity')

all_metrics = {}
cluster_details = {}  # for within-cluster analysis

for i, loc in enumerate(STUDY_REGIONS):
    grid_gdf, step_size_m = grids[loc]
    ntl_values = ntl_data[loc]
    study_itl3 = grid_gdf['ITL3'].unique()
    region_sub = region_gdf[region_gdf['ITL3'].isin(study_itl3)].copy()
    subs_sub = substations_gdf[substations_gdf['ITL3'].isin(study_itl3)].copy().reset_index(drop=True)

    # ── Compute prox2_ntl_landuse_demand ──
    gpm = weighter_registry.create('gpm', config={
        'mode': 'categorical', 'proportion_columns': LU_COLS,
    })
    gpm_res = gpm.compute(grid_gdf, target_gdf=subs_sub)
    grid_gdf = compute_demand(grid_gdf, region_sub, gpm_res, demand_col='landuse_demand')

    ntl_corrector.correct(grid_gdf, region_sub, 'landuse_demand',
                          ntl_values, 'ntl_landuse_demand')

    prox_scores = ProximityCorrector.compute_scores(
        grid_gdf, subs_sub, gamma=GAMMA,
        target_crs=TARGET_CRS, clamp_km=DIST_CLAMP_KM)
    prox_corrector.correct(grid_gdf, region_sub, 'ntl_landuse_demand',
                           prox_scores, 'prox2_ntl_landuse_demand')

    DEMAND_COL = 'prox2_ntl_landuse_demand'

    # ── Voronoi allocation ──
    alloc = allocator_registry.create('voronoi')
    voronoi_res = alloc.allocate(grid_gdf, subs_sub.copy())
    voronoi_result = aggregate_to_substations(
        grid_gdf, subs_sub, voronoi_res.assignment, DEMAND_COL)

    # ── CIVD allocation (load assignment from cache) ──
    coords = np.column_stack([subs_sub.geometry.x.values, subs_sub.geometry.y.values])
    cluster_gdf_loc, _ = do_clustering(coords, method='hdbscan', min_cluster_size=2)

    civd_cache_path = CACHE_DIR / f'{loc}_civd_cache.pickle'
    with open(civd_cache_path, 'rb') as f:
        cache = pickle.load(f)
    civd_assignment = cache['assignment']

    civd_result = aggregate_clustered_to_substations(
        grid_gdf, subs_sub, cluster_gdf_loc, civd_assignment, DEMAND_COL)

    # ── Evaluation ──
    metrics = {
        'voronoi': evaluate_allocation(voronoi_result),
        'civd_equal_split': evaluate_allocation(civd_result),
    }
    all_metrics[loc] = metrics

    # ── Within-cluster analysis: record the actual demand distribution per cluster ──
    cluster_info = []
    for label in sorted(cluster_gdf_loc['cluster_label'].unique()):
        members = cluster_gdf_loc[cluster_gdf_loc['cluster_label'] == label].index
        members = [m for m in members if m < len(subs_sub)]
        if len(members) <= 1:
            continue
        actual_demands = subs_sub.loc[members, 'Demand (MVA)'].values
        allocated_equal = civd_result.loc[members, 'allocated_demand'].values
        cluster_info.append({
            'cluster': label,
            'n_subs': len(members),
            'actual_std': float(np.std(actual_demands)),
            'actual_cv': float(np.std(actual_demands) / np.mean(actual_demands)) if np.mean(actual_demands) > 0 else 0,
            'actual_range': float(actual_demands.max() - actual_demands.min()),
            'equal_split_rmse': float(np.sqrt(np.mean((actual_demands - allocated_equal) ** 2))),
        })
    cluster_details[loc] = cluster_info

    print(f'  [{i+1}/{len(STUDY_REGIONS)}] {loc}: '
          f'Voronoi RMSE={metrics["voronoi"]["rmse"]:.2f}, '
          f'CIVD RMSE={metrics["civd_equal_split"]["rmse"]:.2f}')

print(f'\n{len(STUDY_REGIONS)} regions complete')

In [ ]:
# ─── Voronoi vs CIVD (equal-split) comparison table ───
rows = []
for loc in STUDY_REGIONS:
    m = all_metrics[loc]
    rows.append({
        'region': loc,
        'voronoi_rmse': m['voronoi']['rmse'],
        'civd_rmse': m['civd_equal_split']['rmse'],
        'voronoi_mae': m['voronoi']['mae'],
        'civd_mae': m['civd_equal_split']['mae'],
        'voronoi_corr': m['voronoi']['corr'],
        'civd_corr': m['civd_equal_split']['corr'],
    })

df = pd.DataFrame(rows).set_index('region')
df['rmse_diff'] = df['civd_rmse'] - df['voronoi_rmse']
df['civd_wins'] = df['rmse_diff'] < 0

print('=== RMSE / MAE / Corr Comparison ===')
display(df.round(2))

n_civd_wins = df['civd_wins'].sum()
print(f'\nCIVD (equal-split) RMSE < Voronoi: {n_civd_wins}/{len(df)} regions')
print(f'Mean RMSE — Voronoi: {df["voronoi_rmse"].mean():.2f}, CIVD: {df["civd_rmse"].mean():.2f}')

In [ ]:
# ─── Within-cluster equal-split problem analysis ───
# Shows: the greater the variance in actual demand within a multi-station cluster, the larger the equal-split error

all_clusters = []
for loc in STUDY_REGIONS:
    for c in cluster_details[loc]:
        c['region'] = loc
        all_clusters.append(c)

cdf = pd.DataFrame(all_clusters)

print(f'Total {len(cdf)} multi-station clusters (across {len(STUDY_REGIONS)} regions)')
print(f'  Mean substations per cluster: {cdf["n_subs"].mean():.1f}')
print(f'  Mean within-cluster actual demand CV: {cdf["actual_cv"].mean():.2f}')
print(f'  Mean within-cluster equal-split RMSE: {cdf["equal_split_rmse"].mean():.2f} MVA')

# Group by CV to examine the equal-split error
bins = [0, 0.3, 0.6, 1.0, float('inf')]
labels = ['CV<0.3', '0.3-0.6', '0.6-1.0', 'CV>1.0']
cdf['cv_bin'] = pd.cut(cdf['actual_cv'], bins=bins, labels=labels)
summary = cdf.groupby('cv_bin', observed=True).agg(
    n_clusters=('cluster', 'count'),
    mean_rmse=('equal_split_rmse', 'mean'),
    mean_n_subs=('n_subs', 'mean'),
).round(2)

print('\n=== Within-Cluster Demand Heterogeneity vs. Equal-Split Error ===')
display(summary)
print('\nConclusion: the higher a cluster\'s CV, the larger the equal-split error, so within-cluster weighted allocation is needed')